In [7]:
# ===========================================
# INSTALL DEPENDENCIES
# ===========================================

!pip -q install yfinance fredapi PyPortfolioOpt plotly kaleido \
                 reportlab statsmodels scipy openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 11.6 MB/s eta 0:00:00


In [8]:
# ===========================================
# IMPORT LIBRARIES
# ===========================================

import os
import warnings
import numpy as np
import pandas as pd

import yfinance as yf

import plotly.express as px
import plotly.graph_objects as go

import matplotlib.pyplot as plt

from scipy import stats
import statsmodels.api as sm

from fredapi import Fred

warnings.filterwarnings("ignore")

plt.style.use("ggplot")

In [9]:
# ===========================================
# CREATE PROJECT FOLDERS
# ===========================================

folders = [
    "data",
    "cache",
    "figures",
    "reports",
    "exports",
    "config"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created.")

Project folders created.


In [10]:
# ===========================================
# GLOBAL CONFIGURATION
# ===========================================

START_DATE = "2018-01-01"
END_DATE = None      # None = today's date

BENCHMARK = "SPY"

RISK_FREE_RATE = 0.02

TRADING_DAYS = 252

INITIAL_CAPITAL = 100000

DEFAULT_TICKERS = [
    "AAPL",
    "MSFT",
    "NVDA",
    "AMZN",
    "GOOGL"
]

In [11]:
# ===========================================
# PORTFOLIO WEIGHTS
# ===========================================

weights = np.array([
    0.20,
    0.20,
    0.20,
    0.20,
    0.20
])

print(weights.sum())

1.0


In [12]:
# ===========================================
# VALIDATE INPUTS
# ===========================================

def validate_portfolio(tickers, weights):

    if len(tickers) != len(weights):
        raise ValueError("Number of weights must equal number of tickers.")

    if abs(sum(weights) - 1.0) > 0.0001:
        raise ValueError("Portfolio weights must sum to 1.")

    print("Portfolio validated successfully.")

validate_portfolio(DEFAULT_TICKERS, weights)

Portfolio validated successfully.


In [13]:
# ===========================================
# DOWNLOAD PRICE DATA
# ===========================================

def download_prices(
    tickers,
    start=START_DATE,
    end=END_DATE
):

    prices = yf.download(
        tickers,
        start=start,
        end=end,
        auto_adjust=True,
        progress=False
    )["Close"]

    prices = prices.dropna()

    return prices

In [14]:
prices = download_prices(DEFAULT_TICKERS)

prices.head()

Ticker,AAPL,AMZN,GOOGL,MSFT,NVDA
Date,,,,,
2018-01-02,40.267082,59.450500,53.188866,78.699898,4.922529
2018-01-03,40.260052,60.209999,54.096325,79.066154,5.246499
2018-01-04,40.447063,60.479500,54.306454,79.762047,5.274157
2018-01-05,40.907574,61.457001,55.026566,80.750969,5.318851
2018-01-08,40.755630,62.343498,55.220840,80.833366,5.481823


In [15]:
benchmark = download_prices([BENCHMARK])

benchmark.head()

Ticker,SPY
Date,
2018-01-02,235.954330
2018-01-03,237.446701
2018-01-04,238.447571
2018-01-05,240.036560
2018-01-08,240.475555


In [16]:
print("Assets")

print(prices.columns.tolist())

print()

print("Observations:", len(prices))

print()

print(prices.describe())

Assets
['AAPL', 'AMZN', 'GOOGL', 'MSFT', 'NVDA']

Observations: 2156

Ticker         AAPL         AMZN        GOOGL         MSFT         NVDA
count   2156.000000  2156.000000  2156.000000  2156.000000  2156.000000
mean     145.032717   146.793262   129.436522   272.788091    54.556346
std       74.266408    51.742343    77.311366   126.679372    64.589163
min       33.736992    59.450500    48.800774    77.839195     3.146729
25%       69.715517    95.457375    66.749500   155.988590     6.704080
50%      146.240616   150.000496   114.772289   265.540619    19.530041
75%      193.316566   181.289875   157.240070   391.983109   101.884724
max      340.079987   274.989990   402.379669   538.658569   235.465576


In [17]:
missing = prices.isna().sum()

missing

,0
Ticker,
AAPL,0
AMZN,0
GOOGL,0
MSFT,0
NVDA,0


In [18]:
fig = px.line(
    prices,
    title="Historical Adjusted Prices"
)

fig.show()

In [19]:
prices.to_csv("data/portfolio_prices.csv")

benchmark.to_csv("data/benchmark.csv")

print("Saved.")

Saved.


In [20]:
summary = {
    "Assets": len(DEFAULT_TICKERS),
    "Observations": len(prices),
    "Start Date": prices.index.min(),
    "End Date": prices.index.max(),
    "Benchmark": BENCHMARK
}

pd.DataFrame(summary, index=[0])

,Assets,Observations,Start Date,End Date,Benchmark
0,5,2156,2018-01-02,2026-07-31,SPY


In [21]:
import pandas as pd

prices = pd.read_csv(
    "data/portfolio_prices.csv",
    index_col=0,
    parse_dates=True
)

benchmark = pd.read_csv(
    "data/benchmark.csv",
    index_col=0,
    parse_dates=True
)

In [22]:
print("Portfolio Shape:", prices.shape)
print("Benchmark Shape:", benchmark.shape)

print("\nPortfolio Info")
print(prices.info())

print("\nFirst Five Rows")
display(prices.head())

Portfolio Shape: (2156, 5)
Benchmark Shape: (2156, 1)

Portfolio Info
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2156 entries, 2018-01-02 to 2026-07-31
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    2156 non-null   float64
 1   AMZN    2156 non-null   float64
 2   GOOGL   2156 non-null   float64
 3   MSFT    2156 non-null   float64
 4   NVDA    2156 non-null   float64
dtypes: float64(5)
memory usage: 101.1 KB
None

First Five Rows


,AAPL,AMZN,GOOGL,MSFT,NVDA
Date,,,,,
2018-01-02,40.267082,59.450500,53.188866,78.699898,4.922529
2018-01-03,40.260052,60.209999,54.096325,79.066154,5.246499
2018-01-04,40.447063,60.479500,54.306454,79.762047,5.274157
2018-01-05,40.907574,61.457001,55.026566,80.750969,5.318851
2018-01-08,40.755630,62.343498,55.220840,80.833366,5.481823


In [23]:
missing = prices.isnull().sum()

missing = pd.DataFrame({
    "Missing Values": missing,
    "Percent Missing": (missing / len(prices) * 100).round(2)
})

display(missing)

,Missing Values,Percent Missing
AAPL,0,0.0
AMZN,0,0.0
GOOGL,0,0.0
MSFT,0,0.0
NVDA,0,0.0


In [24]:
prices = prices.loc[~prices.index.duplicated()]
benchmark = benchmark.loc[~benchmark.index.duplicated()]

In [25]:
prices = prices.sort_index()
benchmark = benchmark.sort_index()

In [26]:
daily_returns = prices.pct_change().dropna()

benchmark_returns = benchmark.pct_change().dropna()

In [27]:
display(daily_returns.head())

,AAPL,AMZN,GOOGL,MSFT,NVDA
Date,,,,,
2018-01-03,-0.000175,0.012775,0.017061,0.004654,0.065814
2018-01-04,0.004645,0.004476,0.003884,0.008801,0.005272
2018-01-05,0.011386,0.016163,0.013260,0.012398,0.008474
2018-01-08,-0.003714,0.014425,0.003531,0.001020,0.030641
2018-01-09,-0.000115,0.004676,-0.001274,-0.000680,-0.000270


In [28]:
import numpy as np

log_returns = np.log(prices / prices.shift(1)).dropna()

benchmark_log_returns = np.log(
    benchmark / benchmark.shift(1)
).dropna()

In [29]:
weekly_prices = prices.resample("W").last()

weekly_returns = weekly_prices.pct_change().dropna()

In [30]:
monthly_prices = prices.resample("ME").last()

monthly_returns = monthly_prices.pct_change().dropna()

In [31]:
cumulative_returns = (
    1 + daily_returns
).cumprod()

In [32]:
common_dates = daily_returns.index.intersection(
    benchmark_returns.index
)

daily_returns = daily_returns.loc[common_dates]
benchmark_returns = benchmark_returns.loc[common_dates]

log_returns = log_returns.loc[common_dates]
benchmark_log_returns = benchmark_log_returns.loc[common_dates]

In [33]:
from scipy.stats import zscore

z_scores = daily_returns.apply(zscore)

outliers = (np.abs(z_scores) > 3)

display(outliers.sum())

,0
AAPL,31
AMZN,31
GOOGL,32
MSFT,31
NVDA,22


In [34]:
summary = pd.DataFrame({
    "Mean": daily_returns.mean(),
    "Median": daily_returns.median(),
    "Std Dev": daily_returns.std(),
    "Min": daily_returns.min(),
    "Max": daily_returns.max()
})

display(summary.round(4))

,Mean,Median,Std Dev,Min,Max
AAPL,0.0011,0.0012,0.0193,-0.1286,0.1533
AMZN,0.0009,0.0011,0.0218,-0.1405,0.1532
GOOGL,0.0011,0.0013,0.0196,-0.1163,0.1022
MSFT,0.0010,0.0011,0.0184,-0.1474,0.1551
NVDA,0.0022,0.0027,0.0318,-0.1876,0.2437


In [35]:
import plotly.express as px

fig = px.histogram(
    daily_returns,
    title="Distribution of Daily Returns",
    nbins=100
)

fig.show()

In [36]:
fig = px.line(
    cumulative_returns,
    title="Cumulative Portfolio Returns"
)

fig.show()

In [37]:
import plotly.graph_objects as go

monthly_asset = monthly_returns[monthly_returns.columns[0]].copy()

heatmap_df = monthly_asset.to_frame(name="Return")
heatmap_df["Year"] = heatmap_df.index.year
heatmap_df["Month"] = heatmap_df.index.strftime("%b")

pivot = heatmap_df.pivot_table(
    values="Return",
    index="Year",
    columns="Month"
)

month_order = [
    "Jan","Feb","Mar","Apr","May","Jun",
    "Jul","Aug","Sep","Oct","Nov","Dec"
]
pivot = pivot.reindex(columns=month_order)

fig = go.Figure(
    data=go.Heatmap(
        z=pivot.values,
        x=pivot.columns,
        y=pivot.index,
        colorbar_title="Return"
    )
)

fig.update_layout(
    title=f"Monthly Return Heatmap - {monthly_returns.columns[0]}"
)

fig.show()

In [38]:
daily_returns.to_csv("data/daily_returns.csv")

weekly_returns.to_csv("data/weekly_returns.csv")

monthly_returns.to_csv("data/monthly_returns.csv")

log_returns.to_csv("data/log_returns.csv")

cumulative_returns.to_csv("data/cumulative_returns.csv")

benchmark_returns.to_csv("data/benchmark_returns.csv")

In [39]:
data_summary = pd.DataFrame({
    "Metric": [
        "Assets",
        "Trading Days",
        "Weekly Observations",
        "Monthly Observations"
    ],
    "Value": [
        daily_returns.shape[1],
        len(daily_returns),
        len(weekly_returns),
        len(monthly_returns)
    ]
})

display(data_summary)

,Metric,Value
0,Assets,5
1,Trading Days,2155
2,Weekly Observations,447
3,Monthly Observations,102


In [40]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from scipy.stats import norm

In [41]:
# ==========================================
# LOAD DATA
# ==========================================

daily_returns = pd.read_csv(
    "data/daily_returns.csv",
    index_col=0,
    parse_dates=True
)

benchmark_returns = pd.read_csv(
    "data/benchmark_returns.csv",
    index_col=0,
    parse_dates=True
)


daily_returns.head()

,AAPL,AMZN,GOOGL,MSFT,NVDA
Date,,,,,
2018-01-03,-0.000175,0.012775,0.017061,0.004654,0.065814
2018-01-04,0.004645,0.004476,0.003884,0.008801,0.005272
2018-01-05,0.011386,0.016163,0.013260,0.012398,0.008474
2018-01-08,-0.003714,0.014425,0.003531,0.001020,0.030641
2018-01-09,-0.000115,0.004676,-0.001274,-0.000680,-0.000270


In [42]:
# ==========================================
# PORTFOLIO WEIGHTS
# ==========================================

weights = np.array([
    0.20,
    0.20,
    0.20,
    0.20,
    0.20
])

In [43]:
# ==========================================
# PORTFOLIO RETURNS
# ==========================================

portfolio_returns = (
    daily_returns *
    weights
).sum(axis=1)


portfolio_returns.head()

,0
Date,
2018-01-03,0.020026
2018-01-04,0.005416
2018-01-05,0.012336
2018-01-08,0.009180
2018-01-09,0.000467


In [44]:
# ==========================================
# ANNUALIZED RETURN
# ==========================================

def annualized_return(returns):

    total_return = (
        1 + returns
    ).prod()

    years = len(returns) / 252

    return total_return**(1/years)-1


annualized_return(
    portfolio_returns
)

np.float64(0.3212835410759072)

In [45]:
# ==========================================
# VOLATILITY
# ==========================================

def annualized_volatility(returns):

    return returns.std() * np.sqrt(252)


portfolio_volatility = annualized_volatility(
    portfolio_returns
)


portfolio_volatility

np.float64(0.29063266192438186)

In [46]:
# ==========================================
# DOWNSIDE DEVIATION
# ==========================================

def downside_deviation(
    returns,
    target=0
):

    downside = returns[
        returns < target
    ]

    return downside.std()*np.sqrt(252)


downside_deviation(
    portfolio_returns
)

np.float64(0.2128074181370592)

In [47]:
# ==========================================
# EQUITY CURVE
# ==========================================

initial_value = 100000


equity_curve = (
    1 + portfolio_returns
).cumprod()*initial_value


equity_curve.head()

,0
Date,
2018-01-03,102002.584568
2018-01-04,102555.000959
2018-01-05,103820.134212
2018-01-08,104773.243711
2018-01-09,104822.208606


In [48]:
fig = px.line(
    equity_curve,
    title="Portfolio Equity Curve"
)

fig.show()

In [49]:
# ==========================================
# MAX DRAWDOWN
# ==========================================

def calculate_drawdown(equity):

    peak = equity.cummax()

    drawdown = (
        equity - peak
    ) / peak

    return drawdown


drawdown = calculate_drawdown(
    equity_curve
)


max_drawdown = drawdown.min()

max_drawdown

-0.4182719279713187

In [50]:
fig = px.area(
    drawdown,
    title="Portfolio Drawdown"
)

fig.show()

In [51]:
# ==========================================
# DRAWDOWN DURATION
# ==========================================

def drawdown_duration(drawdown):

    duration = 0
    max_duration = 0

    for value in drawdown:

        if value < 0:
            duration += 1

            max_duration=max(
                max_duration,
                duration
            )

        else:
            duration = 0

    return max_duration


drawdown_duration(drawdown)

391

In [52]:
# ==========================================
# ROLLING VOLATILITY
# ==========================================

rolling_volatility = (
    portfolio_returns
    .rolling(60)
    .std()
    *
    np.sqrt(252)
)


fig = px.line(
    rolling_volatility,
    title="60-Day Rolling Volatility"
)

fig.show()

In [53]:
# ==========================================
# HISTORICAL VAR
# ==========================================

def historical_var(
    returns,
    confidence=0.95
):

    return np.percentile(
        returns,
        (1-confidence)*100
    )


historical_var(
    portfolio_returns
)

np.float64(-0.02966456420051981)

In [54]:
# ==========================================
# PARAMETRIC VAR
# ==========================================

def parametric_var(
    returns,
    confidence=0.95
):

    mu = returns.mean()

    sigma = returns.std()

    z = norm.ppf(
        1-confidence
    )

    return (
        mu + z*sigma
    )


parametric_var(
    portfolio_returns
)

np.float64(-0.028840362313172697)

In [55]:
# ==========================================
# EXPECTED SHORTFALL
# ==========================================

def expected_shortfall(
    returns,
    confidence=0.95
):

    var = historical_var(
        returns,
        confidence
    )

    losses = returns[
        returns <= var
    ]

    return losses.mean()


expected_shortfall(
    portfolio_returns
)

np.float64(-0.0421164161784183)

In [56]:
# ==========================================
# STRESS TEST
# ==========================================

stress_scenarios = {
    "Market Crash": -0.20,
    "Moderate Correction": -0.10,
    "Black Swan": -0.35
}


stress_results = {}

for scenario, shock in stress_scenarios.items():

    stress_results[scenario] = (
        100000*(1+shock)
    )


pd.DataFrame(
    stress_results.items(),
    columns=[
        "Scenario",
        "Portfolio Value"
    ]
)

,Scenario,Portfolio Value
0,Market Crash,80000.0
1,Moderate Correction,90000.0
2,Black Swan,65000.0


In [57]:
# ==========================================
# RISK CONTRIBUTION
# ==========================================

cov_matrix = (
    daily_returns.cov()
    *
    252
)


portfolio_variance = (
    weights.T
    @ cov_matrix
    @ weights
)


marginal_contribution = (
    cov_matrix
    @ weights
)


risk_contribution = (
    weights *
    marginal_contribution
    /
    portfolio_variance
)


pd.DataFrame({
    "Asset": daily_returns.columns,
    "Risk Contribution": risk_contribution
})

,Asset,Risk Contribution
AAPL,AAPL,0.166539
AMZN,AMZN,0.195163
GOOGL,GOOGL,0.174548
MSFT,MSFT,0.170596
NVDA,NVDA,0.293153


In [58]:
# ==========================================
# RISK SUMMARY
# ==========================================

risk_summary = pd.DataFrame({

    "Metric":[
        "Annual Return",
        "Annual Volatility",
        "Maximum Drawdown",
        "Historical VaR",
        "Expected Shortfall"
    ],

    "Value":[

        annualized_return(
            portfolio_returns
        ),

        annualized_volatility(
            portfolio_returns
        ),

        max_drawdown,

        historical_var(
            portfolio_returns
        ),

        expected_shortfall(
            portfolio_returns
        )
    ]

})


risk_summary

,Metric,Value
0,Annual Return,0.321284
1,Annual Volatility,0.290633
2,Maximum Drawdown,-0.418272
3,Historical VaR,-0.029665
4,Expected Shortfall,-0.042116


In [59]:
# ==========================================
# EXPORT
# ==========================================

risk_summary.to_csv(
    "data/risk_summary.csv"
)

drawdown.to_csv(
    "data/drawdown.csv"
)

rolling_volatility.to_csv(
    "data/rolling_volatility.csv"
)